kernel : Python (Pixi)

#### Standardizing Cluster Interpretation and Cell Type Annotation

Public single-cell datasets can be standardized, clustered, and analyzed for differential expression, yet still remain biologically ambiguous when clusters lack curated cell type labels. 
Clusters generated by unsupervised algorithms reflect mathematical similarity in gene expression space, but they do not inherently correspond to known biological cell types or functional states. 
Therefore, cell type annotation is required to assign biological identities to clusters by comparing cluster-level DE and expression patterns against established cell-type marker sets to support interpretable, reproducible downstream analyses.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

series_name = "SCP1038"
clustered_data_location = find_env_dir("CLUSTERED_DATA_LOCATION")
de_analysis_location = find_env_dir("DE_ANALYSIS_LOCATION")

clustered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
de_result = pd.read_csv(os.path.join(de_analysis_location, series_name + "_deseq2.csv"), index_col=0)

de_result.set_index("gene", inplace=True)
de_result.sort_values("log2FoldChange", ascending=False, inplace=True)

LOG_FOLD_CHANGE_THRESHOLD = 2
ADJUSTED_PVALUE_THRESHOLD = 0.01
# The ratio threshold for filtering unique markers
# The ratio is determined by (highest normalized mean count) / (second highest normalized count)
UNIQUE_THRESHOLD = 1.5

/home/sungjune222/projects/scRNA-seq_pipeline/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Identify differentially expressed genes in a specific cluster

In [18]:
cluster = 2
subset = de_result[de_result["cluster"] == cluster]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,cluster,contrast,cluster_key
gene,,,,,,,,,
Gm29135,4.814818,6.934961,0.693153,10.004950,1.449646e-23,3.941172e-22,2,rest,leiden
Gm38505,644.654836,6.069795,0.147463,41.161499,0.000000e+00,0.000000e+00,2,rest,leiden
Col11a1,2036.787038,5.988709,0.138379,43.277692,0.000000e+00,0.000000e+00,2,rest,leiden
Gm44296,4.257735,5.922882,0.629686,9.406082,5.149752e-21,1.230277e-19,2,rest,leiden
Gm13403,45.482211,5.905826,0.236865,24.933266,3.243570e-137,7.320076e-135,2,rest,leiden


Identify clusters where a specific gene is differentially expressed

In [20]:
gene = "Fat2"
subset = de_result.loc[gene]
subset = subset[(subset["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & (subset["padj"] < ADJUSTED_PVALUE_THRESHOLD)]
subset.head()

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,cluster,contrast,cluster_key
gene,,,,,,,,,
Fat2,8.484144,2.108106,0.386436,5.455258,4.890181e-08,5.167594e-07,1,rest,leiden


Identify 

In [ ]:
def filter_unique_markers(adata: AnnData, deseq_results: pd.DataFrame, cluster_key="leiden", unique_threshold=UNIQUE_THRESHOLD):
    candidates = deseq_results[
        (deseq_results["log2FoldChange"] > LOG_FOLD_CHANGE_THRESHOLD) & 
        (deseq_results["padj"] < ADJUSTED_PVALUE_THRESHOLD)
    ].copy()
    candidates.reset_index(inplace=True)
    candidates["cluster"] = candidates["cluster"].astype(str)
    needed_genes = candidates["gene"].unique()

    assert isinstance(adata.X, csr_matrix)
    library_sizes = np.asarray(adata.X.sum(axis=1)).flatten()
    library_sizes[library_sizes == 0] = 1.0
    adata_needed_genes = adata[:, needed_genes].to_memory()
 
    cluster_means_dict = dict()
    TARGET_SUM = 1e4

    for cluster in adata.obs[cluster_key].unique():
        mask = (adata.obs[cluster_key] == cluster).values
        adata_masked = adata_needed_genes[mask]
        libs = library_sizes[mask]
        scaling_factors = TARGET_SUM / libs
        
        assert isinstance(adata_masked.X, csr_matrix)
        row_scale = sparse.diags(scaling_factors)
        X_norm = row_scale @ adata_masked.X
        mean_val = np.asarray(X_norm.mean(axis=0)).flatten()

        cluster_means_dict[cluster] = mean_val
    
    cluster_means = pd.DataFrame.from_dict(cluster_means_dict, orient="index", columns=needed_genes)
    unique_markers = []

    for cluster in candidates["cluster"].unique():
        cluster_sub = candidates[candidates["cluster"] == cluster]
        genes = cluster_sub["gene"].values

        my_means = cluster_means.loc[cluster, genes].values
        rest_means: pd.DataFrame = cluster_means.drop(index=cluster)[genes].max(axis=0).values
        safe_rest_means = np.maximum(rest_means, 1e-9)
        
        ratios = my_means / safe_rest_means
        pass_mask = ratios >= unique_threshold
        
        valid_rows = cluster_sub[pass_mask].copy()
        valid_rows["specificity_ratio"] = ratios[pass_mask]
        valid_rows["my_mean"] = my_means[pass_mask]
        valid_rows["second_best_mean"] = rest_means[pass_mask]

        unique_markers.append(valid_rows)
        
    return pd.concat(unique_markers, ignore_index = True)

In [21]:
markers = filter_unique_markers(clustered_adata, de_result)
markers.head(1)

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,cluster,contrast,cluster_key,specificity_ratio,my_mean,second_best_mean
0,Gnat3,1593.349137,9.641814,0.239707,40.223332,0.0,0.0,12,rest,leiden,138.296265,4.692122,0.033928


In [22]:
cluster_markers = markers[(markers["cluster"] == "15") & (markers["my_mean"] > 10)]
cluster_markers.head()

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,cluster,contrast,cluster_key,specificity_ratio,my_mean,second_best_mean
1768,Slc35f1,7674.373777,7.131791,0.123759,57.626490,0.0,0.0,15,rest,leiden,2.660956,22.163832,8.329275
1771,Cdh19,11433.560007,7.066485,0.179246,39.423349,0.0,0.0,15,rest,leiden,2.603738,32.246948,12.384865
1793,Grik2,8345.531772,6.710352,0.140891,47.628125,0.0,0.0,15,rest,leiden,2.665315,22.970551,8.618324
1812,Sorcs1,4039.018656,6.499450,0.136095,47.756819,0.0,0.0,15,rest,leiden,2.698282,10.828649,4.013164
1831,Chl1,6692.465698,6.373895,0.091269,69.836030,0.0,0.0,15,rest,leiden,2.452946,18.418354,7.508666


In [ ]:
plot.plot_dotplot(clustered_adata, {"ENS":  ["Ptprz1", "Gpm6b", "Sox10", "Gfap", "Plp1", "S100b", "Stmn2", "Chd5", "Tac1", "Vip", "Phox2b", "Tubb2b", "Snap25", "Piezo1", "Piezo2"]})

In [ ]:
cell_marker_genes = {
    "Musculature": [
        "Kit", "Lrig1", "Prkg1", "Pcdh7", "Foxp2", 
        "Cpe", "Meis2", "Acta2", "Tagln", "Mylk", 
        "Actg2", "Myh11"
    ],
    "Vasculature": [
        "Kdr", "Ecscr", "Cldn5", "Pecam1", "Lyve1", "Flt1"
    ],
    "ENS": [
        "Snap25", "Tubb2b", "Phox2b", "Vip", "Tac1", 
        "Chd5", "Stmn2", "S100b", "Plp1", "Gfap", 
        "Sox10", "Gpm6b", "Ptprz1"
    ],
    "Immune": [
        "Ptprc", "Ikzf1", "Ikzf3", "Arhgdib", "Coro1a", 
        "Ccr5", "Cd4", "Itgax", "Cd8a", "Cd3g", "Cd3d"
    ],
    "Mesenchymal": [
        "Vim", "Col1a1", "Cd81", "Cd34", "Col1a2", 
        "Dcn", "Wt1", "Thy1", "Pdpn", "Col3a1", 
        "Myrf", "Bmp5", "Pdgfra"
    ]
}
plot.plot_dotplot(clustered_adata, cell_marker_genes)

In [ ]:
leiden_to_celltype = {
    "0": "??",
    "1": "??",
    "2": "ENS",
    "3": "??",
    "4": "??",
    "5": "??",
    "6": "??",
    "7": "??",
    "8": "??",
    "9": "Immune",
    "10": "??",
    "11": "??",
    "12": "??",
    "13": "ENS",
    "14": "??",
    "15": "ENS",
    "16": "Vasculature",
    "17": "Vasculature",
    "18": "??",
    "19": "??",
    "20": "??",
    "21": "??",
    "22": "??",
    "23": "??",
    "24": "??",
    "25": "??",
    "26": "??",
    "27": "??",
    "28": "??",
    "29": "Vasculature",
    "30": "??",
    "31": "Immune",
    "32": "??",
    "33": "??",
    "34": "??",
    "35": "??",
    "36": "??",
    "37": "??",
    "38": "??",
    "39": "??",
    "40": "??",
    }